In [2]:
import os
os.chdir("..")

In [3]:
# Constants
# COMPUTE_DTYPE = torch.bfloat16
DEVICE = 'cuda'
MODEL_ID = "Qwen/QwQ-32B"
MODEL_ID = "Qwen/Qwen2.5-32B"
# MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B"
model_type = "qwq"
model_name = "qwq-32b"
# model_name = "llama-3_3-nemotron-super-49b-v1"
# model_name = "deepseek-qwen-32b"
LAYERS = list(range(50))
N_ROWS = 20
CONT_SIZE = 100

In [4]:
from utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN, THINK_START_TOKEN, DOMAIN_PHRASES

tokenizer = initialize_tokenizer(MODEL_ID)

In [5]:
import torch

def extract_all_phrase_positions(tokens, phrase, cot_only=False):
    """Find all phrase positions in the tokens"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase, add_special_tokens=False),
        tokenizer.encode(" " + phrase.capitalize(), add_special_tokens=False),
        tokenizer.encode("\n" + phrase, add_special_tokens=False)[1:],
        tokenizer.encode("\n" + phrase.capitalize(), add_special_tokens=False)[1:],
        tokenizer.encode("\n\n" + phrase, add_special_tokens=False)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize(), add_special_tokens=False)[1:],
    ]
    positions = set()

    if cot_only:
        think_token_id = 151667  # This should be THINK_TOKEN or an appropriate constant
        start_pos = torch.where(tokens == think_token_id)[0]
        if len(start_pos) > 0:
            start_mask = torch.arange(tokens.shape[0]) >= start_pos[0]
        else:
            start_mask = torch.ones_like(tokens).bool()

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        if cot_only:
            presence_mask = presence_mask * start_mask

        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        for p in (torch.where(presence_mask)[0]).tolist():
            positions.add(
                tuple([p-1, p + len(phts)])
            )
    
    return sorted(list(positions))

In [6]:
from pathlib import Path
import json
from datasets import load_dataset

CUR_DIR = Path(".").absolute()

def load_dataset_from_file(domain_name, task_name):
    """Load dataset from a JSON file"""
    prompt_dir = CUR_DIR / Path(f"./cot-planning/results/{domain_name}/{model_name}-greedy")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

def load_datasets(domain_names, dataset_types):
    """Load datasets and prepare evaluation results"""
    task_name = "plan_generation_po"
    
    # Load evaluation results
    eval_results = [
        load_dataset_from_file(domain_name, task_name)["instances"] 
        for domain_name in domain_names
    ]
    eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]
    
    # Load datasets
    datasets = [
        load_dataset(f"dmitriihook/{model_name}-planning-{dataset_type}-greedy")["train"]
        # load_dataset(f"dmitriihook/llama-3_3-nemotron-super-49b-v1-planning-{dataset_type}-greedy")["train"]
        for dataset_type in dataset_types
    ]
    
    # Load label datasets
    # label_datasets = [
    #     load_dataset(f"dmitriihook/{domain_name.replace('_', '-')}-qwq-reasoning-parts-exploration")["train"]
    #     for domain_name in domain_names
    # ]
    
    # label_datasets = [
    #     {x["index"]: x for x in ld} for ld in label_datasets
    # ]
    
    label_datasets = [
        None for _ in domain_names
    ]
    
    return eval_results, datasets, label_datasets


In [7]:
# Set domain names and dataset types for clean domain
domain_names = ["blocksworld_mystery_2"]
dataset_types = ["mystery-2-24k"]

# Get domain-specific phrases
clean_domain_phrases = DOMAIN_PHRASES["mystery_2"]

# Extract action and predicate phrases
clean_action_phrases = list(clean_domain_phrases["actions"].values())
clean_predicate_phrases = list(clean_domain_phrases["predicates"].values())
clean_phrases = clean_action_phrases + clean_predicate_phrases

# try:
# Load datasets
eval_results, datasets, label_datasets = load_datasets(domain_names, dataset_types)

# Find label positions
# label_positions = [
#     find_label_positions(ld, dataset) 
#     for ld, dataset in zip(label_datasets, datasets)
# ]

label_positions = [
    None for _ in datasets
]

# Collect correct IDs
# correct_ids = [
#     collect_correct_ids(er) for er in eval_results
# ]

# print(len(correct_ids[0]))
print(
    datasets[0][0]
)

row = datasets[0][0]
generation = row["generation"]
text = generation.split("</think>")[0]
tokens = tokenize_blocksworld_generation(tokenizer, row, text, model_type=model_type)[0]
end_pos = len(tokens)

extracted_positions = {
    phrase: [] for phrase in clean_phrases
}

for phrase in clean_phrases:
    positions = extract_all_phrase_positions(tokens, phrase)
    positions = [p for p in positions if p[0] < end_pos]
    extracted_positions[phrase].append(positions)

print(extracted_positions)

{'query': "I am playing with a set of objects. Here are the actions I can do\n\n   Illuminate object\n   Distill object from another object\n   Silence object\n   Divest object from another object\n\nI have the following restrictions on my actions:\n    To perform Illuminate action, the following facts need to be true: Essence object, Aura object, Nexus.\n    Once Illuminate action is performed the following facts will be true: Pulse object.\n    Once Illuminate action is performed the following facts will be false: Essence object, Aura object, Nexus.\n    To perform Silence action, the following facts need to be true: Pulse object.\n    Once Silence action is performed the following facts will be true: Essence object, Aura object, Nexus.    \n    Once Silence action is performed the following facts will be false: Pulse object.\n    To perform Distill action, the following needs to be true: Essence other object, Pulse object.\n    Once Distill action is performed the following will be 

In [14]:
"aura" in text

True

In [11]:
phrase = "silence"

phrase_tokens = [
    tokenizer.encode(" " + phrase),
    tokenizer.encode(" " + phrase.capitalize()),
    tokenizer.encode("\n" + phrase)[1:],
    tokenizer.encode("\n" + phrase.capitalize())[1:],
    tokenizer.encode("\n\n" + phrase)[1:],
    tokenizer.encode("\n\n" + phrase.capitalize())[1:],
]


In [12]:
phrase_tokens

[[21162], [68088], [34804, 763], [27571, 763], [34804, 763], [27571, 763]]

In [59]:
tokenizer.encode(" \n\silence")

[151646, 715, 32407, 321, 763]

In [64]:
[
    tokenizer.decode(x) for x in tokens
]

['<｜begin▁of▁sentence｜>',
 '<｜User｜>',
 'I',
 ' am',
 ' playing',
 ' with',
 ' a',
 ' set',
 ' of',
 ' objects',
 '.',
 ' Here',
 ' are',
 ' the',
 ' actions',
 ' I',
 ' can',
 ' do',
 '\n\n',
 '  ',
 ' Illuminate',
 ' object',
 '\n',
 '  ',
 ' Dist',
 'ill',
 ' object',
 ' from',
 ' another',
 ' object',
 '\n',
 '  ',
 ' Silence',
 ' object',
 '\n',
 '  ',
 ' Div',
 'est',
 ' object',
 ' from',
 ' another',
 ' object',
 '\n\n',
 'I',
 ' have',
 ' the',
 ' following',
 ' restrictions',
 ' on',
 ' my',
 ' actions',
 ':\n',
 '   ',
 ' To',
 ' perform',
 ' Illuminate',
 ' action',
 ',',
 ' the',
 ' following',
 ' facts',
 ' need',
 ' to',
 ' be',
 ' true',
 ':',
 ' Essence',
 ' object',
 ',',
 ' Aura',
 ' object',
 ',',
 ' Nexus',
 '.\n',
 '   ',
 ' Once',
 ' Illuminate',
 ' action',
 ' is',
 ' performed',
 ' the',
 ' following',
 ' facts',
 ' will',
 ' be',
 ' true',
 ':',
 ' Pulse',
 ' object',
 '.\n',
 '   ',
 ' Once',
 ' Illuminate',
 ' action',
 ' is',
 ' performed',
 ' the',
 ' foll

In [65]:
row

{'query': "I am playing with a set of objects. Here are the actions I can do\n\n   Illuminate object\n   Distill object from another object\n   Silence object\n   Divest object from another object\n\nI have the following restrictions on my actions:\n    To perform Illuminate action, the following facts need to be true: Essence object, Aura object, Nexus.\n    Once Illuminate action is performed the following facts will be true: Pulse object.\n    Once Illuminate action is performed the following facts will be false: Essence object, Aura object, Nexus.\n    To perform Silence action, the following facts need to be true: Pulse object.\n    Once Silence action is performed the following facts will be true: Essence object, Aura object, Nexus.    \n    Once Silence action is performed the following facts will be false: Pulse object.\n    To perform Distill action, the following needs to be true: Essence other object, Pulse object.\n    Once Distill action is performed the following will be 

In [68]:
tokens

tensor([151646, 151644,     40,  ...,   3119,    624, 151643])

In [67]:
tokenizer.apply_chat_template(
    conversation=row["distilabel_metadata"]["raw_input_text_generation_0"]
)

[151646,
 151644,
 40,
 1079,
 5619,
 448,
 264,
 738,
 315,
 6171,
 13,
 5692,
 525,
 279,
 6168,
 358,
 646,
 653,
 271,
 256,
 7518,
 1633,
 198,
 256,
 27604,
 483,
 1633,
 504,
 2441,
 1633,
 198,
 256,
 68088,
 1633,
 198,
 256,
 8765,
 477,
 1633,
 504,
 2441,
 1633,
 271,
 40,
 614,
 279,
 2701,
 16869,
 389,
 847,
 6168,
 510,
 262,
 2014,
 2736,
 7518,
 1917,
 11,
 279,
 2701,
 13064,
 1184,
 311,
 387,
 830,
 25,
 83770,
 1633,
 11,
 62286,
 1633,
 11,
 40022,
 624,
 262,
 9646,
 7518,
 1917,
 374,
 10660,
 279,
 2701,
 13064,
 686,
 387,
 830,
 25,
 49249,
 1633,
 624,
 262,
 9646,
 7518,
 1917,
 374,
 10660,
 279,
 2701,
 13064,
 686,
 387,
 895,
 25,
 83770,
 1633,
 11,
 62286,
 1633,
 11,
 40022,
 624,
 262,
 2014,
 2736,
 68088,
 1917,
 11,
 279,
 2701,
 13064,
 1184,
 311,
 387,
 830,
 25,
 49249,
 1633,
 624,
 262,
 9646,
 68088,
 1917,
 374,
 10660,
 279,
 2701,
 13064,
 686,
 387,
 830,
 25,
 83770,
 1633,
 11,
 62286,
 1633,
 11,
 40022,
 13,
 1066,
 262,
 9646,
 6